# Brain-expressed CNV overlap with SCHEMA and schizophrenia GWAS

Use `CNV_NETWORK=PCNET2.2_brain` (default) or `STRING_high_conf_brain`.
HPA expression and region files are frozen in `input/expression/HPA_brain/`. Retain genes with **TPM > 1 in any of the 13 brain regions**, using the TPM column (not nTPM or pTPM). Exact unambiguous HPA gene symbols map to network Entrez identifiers. Unmapped or ambiguous genes are excluded.
The induced graph is restricted to its largest connected component **before propagation**; seeds and null references use that universe. Sets with fewer than 3 retained seeds are excluded. Inspect `data/network_configs/<network>/brain_filter.json` and run seed QC for losses.
Caches are under `data/cache/PCNET2.2_brain/` or `data/cache/STRING_high_conf_brain/`; results are under the corresponding `results/<network>/`. Original workflows retain their own paths. Completed results are reused by content identity.
Downloads are identified by checksums; the unversioned HPA URL is not assumed to be the historical v24 release.


In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cnv_overlap.pipeline import AnalysisConfig, run
from cnv_overlap.visualization import pair_heatmap, size_scatter


ROOT = Path.cwd()
if ROOT.name == "notebooks":  # opened from notebooks/ in Jupyter
    ROOT = ROOT.parent
if not (ROOT / "configs").is_dir():
    raise RuntimeError("Start this notebook with the project root as working directory")
CONFIG_PATH = Path(os.environ.get("CNV_CONFIG", ROOT / "configs/cnv_disorder_overlap.json"))
import sys
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from network_inputs import load_network_inputs
# Select PCNET2.2_brain or STRING_high_conf_brain.
NETWORK = os.environ.get("CNV_NETWORK", "PCNET2.2_brain")
if not NETWORK.endswith("_brain"):
    NETWORK += "_brain"
# Minimum seeds a gene set needs inside the selected network; smaller sets are excluded.
MIN_SEEDS = 3
spec, root, inputs, provenance = load_network_inputs(CONFIG_PATH, NETWORK, min_seeds=MIN_SEEDS)
RESULTS_ROOT = (root / spec["output_root"]).parent
print("Network:", NETWORK or "custom snapshot", "Results:", RESULTS_ROOT, "Cache:", root / spec["cache_dir"])
config = AnalysisConfig.from_dict(spec["analysis"])
if config.comparisons != "cnv_disorder":
    raise ValueError("This notebook requires analysis.comparisons=cnv_disorder")
A, nodes, symbols, entities, coordinates, regions = inputs
print(f"{len(nodes):,} nodes, {A.nnz // 2:,} edges; {len(entities)} gene sets")
print("Primary: overlap size; NetColoc normal-tail p-values; overlap policy:", config.overlap)
print("Gene-label permutations:", config.permutation_reps)

from notebook_analysis import SCZ_TARGETS, disorder_gene_partition


In [ ]:
# One run gives one BH family across every CNV–target pair for each statistic.
entities = {name: entity for name, entity in entities.items()
            if entity["kind"] == "cnv" or (entity["kind"] == "target" and name in SCZ_TARGETS)}
if not any(e["kind"] == "cnv" for e in entities.values()) or not any(e["kind"] == "target" for e in entities.values()):
    raise ValueError("Need CNVs and at least one target set")
HAS_SCZ = any(name in entities and entities[name]["kind"] == "target" for name in SCZ_TARGETS)
FAMILY_INPUTS = {"combined": (A, nodes, symbols, entities, coordinates, regions)}
RUNS = {"combined": run(*FAMILY_INPUTS["combined"], config=config,
    output_root=root/spec["output_root"]/"combined", cache_dir=root/spec["cache_dir"],
    provenance={**provenance, "question_family": "combined",
                "included_targets": list(SCZ_TARGETS),
                "analysis_status": "brain-expressed SCHEMA/GWAS comparisons"})}
for family, run_dir in RUNS.items():
    assert json.loads((run_dir / "manifest.json").read_text())["status"] == "complete"
    print(family, run_dir)
    print((run_dir / "summary.md").read_text())
from cnv_overlap.provenance import digest
FIGURES = RESULTS_ROOT / "figures/disorder_structure" / digest({k: v.name for k, v in RUNS.items()})[:20]
FIGURES.mkdir(parents=True, exist_ok=True)
if not HAS_SCZ:
    print("No schema/gwaspri sets: these custom inputs have secondary comparisons only.")


## Input QC and tested hypotheses

Inspect seed losses, coordinate coverage and the hypothesis list. Coordinates document CNV content; all retained network genes are eligible for degree-matched references.
The SCZ GWAS prioritized genes and comparator Catalog genes have different ascertainment; see `input/processed/source_manifest.json`.


In [ ]:
for family, run_dir in RUNS.items():
    print("Family:", family)
    display(pd.read_csv(run_dir / "seed_qc.tsv", sep="\t"))
    display(pd.read_csv(run_dir / "seed_overlap.tsv", sep="\t").set_index("entity"))
    print("Planned comparisons:", len(pd.read_csv(run_dir / "hypotheses.tsv", sep="\t")))


## CNV–SCHEMA and CNV–GWAS profiles

One BH family covers all retained CNV–SCHEMA and CNV–GWAS pairs, separately for each null and statistic. Other disorder targets are excluded. Stars indicate BH q < 0.05 for overlap size; empirical q-values are sensitivity diagnostics.


In [ ]:
from cnv_overlap.statistics import bh
frames = []
for family, run_dir in RUNS.items():
    frame = pd.read_csv(run_dir / "pair_tests.tsv", sep="\t").assign(question_family=family)
    frame["target_group"] = np.where(frame.b.isin(SCZ_TARGETS), "scz", "secondary")
    frame["q_empirical"] = np.nan
    for _, idx in frame.groupby(["null", "family", "statistic"]).groups.items():
        frame.loc[idx, "q_empirical"] = bh(frame.loc[idx, "p_empirical"])
    frames.append(frame)
    for metric in ("ratio", "excess", "observed", "z"):
        fig = pair_heatmap(run_dir, family="locus_target", fdr=config.fdr, metric=metric)
        fig.savefig(FIGURES / f"{family}_{metric}.png", dpi=200, bbox_inches="tight")
        fig.savefig(FIGURES / f"{family}_{metric}.pdf", bbox_inches="tight")
        plt.show()
    # Brain analysis includes SCHEMA/GWAS only; no non-SCZ heatmap applies.
    fig = size_scatter(run_dir, family="locus_target", fdr=config.fdr)
    fig.savefig(FIGURES / f"{family}_sizes.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIGURES / f"{family}_sizes.pdf", bbox_inches="tight")
    plt.show()
tests = pd.concat(frames, ignore_index=True)
primary = tests[(tests["null"] == "netcoloc") & (tests.statistic == "size")].copy()
primary.to_csv(FIGURES / "all_primary_pairs.tsv", sep="\t", index=False)
with pd.option_context("display.max_rows", None):
    display(primary.sort_values(["question_family", "a", "b"])[["question_family", "a", "b",
        "observed", "null_mean", "excess", "p", "q", "p_empirical", "q_empirical", "status"]])


## Brain-focused Enrichr analysis of CNV × SCHEMA and CNV × GWAS pairs

Each query is the shared, thresholded candidate-gene set of a CNV × SCHEMA or CNV × GWAS pair. **Every CNV is tested against both targets**; pairs with NetColoc size BH q < config.fdr are marked with **\*** next to the CNV name. Enrichment in an unmarked row is descriptive only: that CNV's network overlap with the target was not significant. CNV × CNV pairs are shown in the convergence notebook.

Use the **Enrichr custom-background API (Speedrichr)** with GO Biological Process 2026 and SynGO 2024. Reactome uses the **official June 2026 GMT** (`input/annotations/ReactomePathways.gmt`, the same release as the rest of the project); Enrichr only offers Reactome 2024, so the identical one-sided hypergeometric test is computed locally with the same background (it matches Enrichr's p-values to within 1%). SynGO provides brain-focused synaptic annotation; GO and Reactome are tested in the same brain-network background. This is not a claim that every GO/Reactome term is brain-specific.

The background is the actual propagated brain-network universe; shared seeds are removed only for `overlap=remove`. Terms need ≥5 background genes. Enrichr p-values are BH-corrected **across all tested pairs and all three libraries**, including zero-hit terms. Enrichr's own per-library adjusted p-values are also retained. Enrichment is descriptive after pair/network selection, not selection-adjusted evidence.

Two plots: CNV × SCHEMA and CNV × GWAS, one row per CNV. Columns, per library, are every row's 2 strongest significant terms, filled to 12 with the strongest terms overall (ordered by best combined q), so a strong pair cannot hide the others; columns are coloured by library; dot size is −log10(combined BH q). Blank positions have no significant enrichment.

**Term collapsing (orsum 1.8.0).** For each plot and library, orsum receives every row's significant terms as one ranked list and picks representative terms jointly across rows, using the full annotation sets. Each column is a representative; **(+n)** counts the further terms it represents, and a dot shows the best original q among them. q-values are never recomputed. Reactome collapses strongly because its gene sets are nested; Enrichr's GO BP sets are not propagated up the GO hierarchy, so GO collapses only modestly. Uncollapsed plots are saved as `*_all_terms.png`, and `*_term_mapping.tsv` lists every representative with the terms it covers. Raw API responses, downloaded GMTs, full result tables, coverage, query genes and backgrounds are cached alongside PNG/PDF figures. The first run submits query/background gene symbols to Enrichr; subsequent runs reuse cached responses.

In [ ]:
from brain_pair_enrichment import enrich_pairs
from IPython.display import Image
BRAIN_PAIR_ENRICHMENT = enrich_pairs(
    RUNS["combined"], annotation_dir=root / "input/annotations",
    output_root=RESULTS_ROOT / "brain_pair_enrichment_enrichr" / config.comparisons,
    fdr=config.fdr, backend="enrichr",
)
print((BRAIN_PAIR_ENRICHMENT / "summary.md").read_text())
display(pd.read_csv(BRAIN_PAIR_ENRICHMENT / "tested_pairs.tsv", sep="\t")[["a", "b", "pair_group", "q", "significant"]])
for figure in ["enrichment_schema", "enrichment_gwas"]:
    display(Image(filename=str(BRAIN_PAIR_ENRICHMENT / f"{figure}.png")))
display(pd.read_csv(BRAIN_PAIR_ENRICHMENT / "enrichment_significant.tsv", sep="\t").head(40))
print("Complete brain enrichment results:", BRAIN_PAIR_ENRICHMENT)


## All-pair gene candidates and descriptive recurrence

Candidate genes meet fixed joint-score thresholds. All candidates are retained, irrespective of pair q-value. Recurrence below counts CNVs and targets among these thresholded pairs; it has no gene-level significance claim. The multi-CNV notebook tests convergence independently of this pair-selection summary.

In [ ]:
display(tests[tests.statistic != "size"].sort_values(["question_family", "statistic", "q"]).head(30))
candidates = pd.concat([pd.read_csv(p / "candidate_genes.tsv", sep="\t").assign(question_family=f)
                        for f, p in RUNS.items()], ignore_index=True)
candidates.to_csv(FIGURES / "all_candidate_genes.tsv", sep="\t", index=False)
display(candidates.head(40))
recurrence = candidates.groupby(["entrez", "symbol"]).agg(n_pairs=("a", "size"),
    n_cnvs=("a", "nunique"), n_targets=("b", "nunique")).reset_index()
recurrence.to_csv(FIGURES / "all_pair_descriptive_recurrence.tsv", sep="\t", index=False)
display(recurrence.sort_values(["n_cnvs", "n_pairs"], ascending=False).head(30))


## Descriptive communities

Each connected component is partitioned separately. Distinct components are never pooled into a fictitious community zero. Recurrence, degree and community membership describe the candidate network; they are not independent replication or significance tests.

In [ ]:
from cnv_overlap.interpretation import candidate_network
G, membership = candidate_network(A, nodes, candidates)
membership["symbol"] = membership.entrez.map(symbols)
membership.to_csv(FIGURES / "candidate_communities.tsv", sep="\t", index=False)
print(f"{G.number_of_nodes()} candidate nodes, {G.number_of_edges()} edges")
display(membership.sort_values(["community", "degree_in_candidate_network"], ascending=[True, False]).head(40))


## Offline annotation and rewiring robustness

Run annotation with explicit GMT snapshots using `cnv-overlap annotate`; query/background use the same selected network universe. Enrichment is descriptive ORA after network selection. Brain-filtered inputs and the retained network universe are used throughout this notebook.

In [ ]:
from notebook_analysis import export_versioned_report
GMT_PATHS = [root / "input/annotations" / name for name in
             ["GO_Biological_Process_2026.gmt", "ReactomePathways.gmt"]]
if not all(path.exists() for path in GMT_PATHS):
    print("Annotation snapshots missing; exporting figures and networks only.")
    GMT_PATHS = []
REPORTS = {}
for family, run_dir in RUNS.items():
    REPORTS[family] = export_versioned_report(run_dir, FAMILY_INPUTS[family],
        output_dir=RESULTS_ROOT / "reports" / family / run_dir.name,
        gmt_paths=GMT_PATHS, fdr=config.fdr)
    print(family, REPORTS[family])
    if GMT_PATHS:
        annotation = pd.read_csv(REPORTS[family] / "enrichment/descriptive_ora.tsv", sep="\t")
        selected_pairs = primary.loc[(primary.question_family == family) & (primary.q < config.fdr), ["a", "b"]]
        displayed = annotation.merge(selected_pairs, on=["a", "b"])
        display(displayed[displayed.q_all_pair_terms < config.fdr].sort_values("q_all_pair_terms").head(40))
